**`inspect_entity_statistics`**

A generic diagnostic overview of any recipe's saved output.

Works at any pipeline stage -- ingest, harmonize, enrich, or curate --
and for any entity type. It reports:

- Column completeness
- Schema conformance against the attribute registry
- Categorical value distributions
- Geometry validity
- Per-admin-unit coverage maps
- An optional multi-recipe comparison

This notebook is read-only. It never writes to disk.

# Configure

In [ ]:
import argparse

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
from matplotlib.lines import Line2D

import openplaces as op
from openplaces.core.attribute_registry import get_attributes
from openplaces.core.schema import AdminId
from openplaces.diagnostics import (
    check_geometry,
    profile_columns,
    summarize_categoricals,
)
from openplaces.recipe import (
    get_recipe_by_id,
    get_save_admin_level,
    resolve_attribute_name,
)
from openplaces.viz.colors import (
    DATA_AVAILABILITY_MISSING_COLOR,
    get_diverging_colormap,
)
from openplaces.viz.maps import show_random_entity

In [ ]:
parser = argparse.ArgumentParser(
    description='Generic diagnostic overview of a recipe output'
)
parser.add_argument(
    '--recipe_id',
    required=True,
    help='Recipe ID to inspect (e.g. "US_parcel-spine-2026")',
)
parser.add_argument(
    '--admin_ids',
    nargs='*',
    required=True,
    help='Admin unit IDs to load and profile (e.g. "US-NC-BL")',
)
parser.add_argument(
    '--compare_recipe_ids',
    nargs='*',
    default=[],
    help=(
        'Optional additional recipe IDs sharing overlapping columns '
        'to cross-compare against (e.g. a statewide fallback source '
        'vs. a county-specific one)'
    ),
)
parser.add_argument(
    '--nmax_categories',
    type=int,
    default=100,
    help='Maximum distinct values to report per categorical column',
)
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US_parcel-spine-2026 '
    # '--admin_ids US-NC-BL US-NC-CW US-NC-RB US-NC-PD '  # newly added NC counties (pilot)
    '--admin_ids US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO '
    '--verbose '
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

# Resolve recipe

In [ ]:
recipe = get_recipe_by_id(args.recipe_id)
entity = recipe.get('entity')
entity_type = str(entity.entity_type) if entity is not None else None
stage = recipe.get('stage', 'ingest')
save_level = get_save_admin_level(recipe)

print(f'entity_type={entity_type!r}  stage={stage!r}  save_admin_level={save_level}')

# Expand/truncate requested admin_ids to the recipe's own save level,
# the same way Harmonizer.__init__ resolves admin_ids for a recipe.
admin_ids = []
for aid_str in args.admin_ids:
    aid = AdminId(aid_str)
    if aid.get_level() < save_level:
        admin_ids += op.get_admin_ids(save_level, admin_id=aid)
    elif aid.get_level() == save_level:
        admin_ids.append(str(aid))
    else:
        admin_ids.append(str(AdminId(*aid.levels[:save_level])))
admin_ids = list(dict.fromkeys(admin_ids))
print(', '.join(admin_ids))

# Load data

In [ ]:
per_admin = {}
for admin_id in admin_ids:
    gdf = op.get_entities(args.recipe_id, admin_id=admin_id, geom=True, missing='warn')
    if gdf is None or len(gdf) == 0:
        continue
    per_admin[admin_id] = gdf
    if args.verbose:
        print(f'  {admin_id}: {len(gdf):,d} rows')

if not per_admin:
    raise ValueError(f'No data loaded for {args.recipe_id} at {admin_ids}.')

combined = pd.concat(per_admin.values())

# inspect_table transposes the frame, so each column becomes a display
# row -- without raising max_rows, pandas' default (60) truncates a
# wide table like this one before it ever reaches the columns at the
# end.
pd.options.display.max_rows = combined.shape[1] + 5
op.inspect_table(combined)

# Column completeness

A value is *meaningfully populated* when it's:

- Non-null, and
- Not a placeholder (`0` or `''`)

See `profile_columns` for the exact rule.

Why it matters: a placeholder can look complete but isn't usable. An
all-`0` `year_built` column, for example, is 100% non-null yet carries
no real data.

Only the meaningfully-populated fraction is shown below, and mapped
further down. The raw non-null fraction is never shown as if it were
data.

In [ ]:
profile = profile_columns(combined, entity_type=entity_type, stage=stage)
profile

## Violinplot

In [ ]:
# Per-admin-unit completeness, reused by both this plot and the coverage
# map below.
per_admin_profile = {
    admin_id: profile_columns(gdf, entity_type=entity_type, stage=stage)[
        'frac_meaningful'
    ]
    for admin_id, gdf in per_admin.items()
}
coverage = pd.DataFrame(per_admin_profile).T  # admin_id x column

# A column absent from one admin unit's schema entirely (not merely
# unpopulated) still means 0% of that admin unit's rows are usable for
# it -- pd.DataFrame(...).T otherwise leaves those cells NaN, which
# mean()/std() silently skip. Left unfixed, a column that only ever
# appeared in a single county (e.g. subdivision_code) would look like
# it has zero spread instead of near-zero coverage -- and, in the map
# below, a county missing only this one column would be mistaken for a
# county with no recipe output at all.
coverage = coverage.fillna(0.0)


def _common_admin_scope(ids: list[str]) -> str:
    """Longest admin-id prefix shared by every id in *ids*."""
    levels = list(zip(*(aid.split('-') for aid in ids), strict=True))
    common = []
    for level in levels:
        if len(set(level)) > 1:
            break
        common.append(level[0])
    return '-'.join(common) if common else 'multiple regions'


scope = _common_admin_scope(admin_ids)

# Sort/color by the plain (unweighted) mean across admin units -- each
# admin unit counts once, regardless of its row count, so a huge county
# can't drown out a small one's coverage.
order = coverage.mean().sort_values().index
fig, ax = plt.subplots(figsize=(6, max(3, 0.2 * len(order))))
if len(per_admin) > 1:
    # One violin per column, showing the spread of meaningful-completeness
    # across admin units, not just its average -- a column that is 80%
    # complete everywhere and one that is 40%/100% split by county both
    # average to 60%, but only the second is worth a closer look. Fill
    # color encodes the same unweighted mean used for the sort order.
    parts = ax.violinplot(
        [coverage[c].to_numpy() * 100 for c in order],
        positions=range(len(order)),
        orientation='horizontal',
        widths=0.8,
        showmeans=True,
        showmedians=True,
    )
    cmap = get_diverging_colormap('ember_ocean')
    norm = plt.Normalize(vmin=0, vmax=100)
    for body, column in zip(parts['bodies'], order, strict=True):
        color = cmap(norm(coverage[column].mean() * 100))
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.8)

    # The min-max span is a hairline reference, not a data mark in its
    # own right -- half the weight of everything else on the violin.
    for key in ('cbars', 'cmins', 'cmaxes'):
        parts[key].set_color('#333333')
    parts['cbars'].set_linewidth(0.5)

    # Mean and median often sit close together on a skewed distribution
    # -- style, not just position, needs to tell them apart.
    parts['cmeans'].set_color('#333333')
    parts['cmeans'].set_linestyle('dashed')
    parts['cmeans'].set_linewidth(1.0)
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linestyle('solid')
    parts['cmedians'].set_linewidth(1.0)
    ax.legend(
        handles=[
            Line2D([0], [0], color='#333333', linestyle='dashed', label='mean'),
            Line2D([0], [0], color='black', linestyle='solid', label='median'),
        ],
        loc='lower right',
        fontsize='small',
        frameon=False,
    )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.02, aspect=30)
    cbar.set_label('mean % meaningfully populated')
    cbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    xlabel = 'meaningfully populated, across admin units'
else:
    admin_id = next(iter(per_admin))
    ax.barh(range(len(order)), coverage.loc[admin_id, order] * 100, color='#3778bf')
    xlabel = 'meaningfully populated'

ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.set_xlim(0, 100)
ax.margins(y=0.01)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel(xlabel)
ax.set_title(
    f'{args.recipe_id}: column completeness\n({len(per_admin)} admin units in {scope})'
)
ax.grid(axis='x', color='#d8d8d8', linewidth=0.6)
ax.set_axisbelow(True)

# Duplicate the percentage ticks on top -- useful on a figure this tall,
# where the bottom axis labels are far from the upper rows.
ax_top = ax.secondary_xaxis('top')
ax_top.xaxis.set_major_formatter(mtick.PercentFormatter())

mismatched = profile[profile['dtype_mismatch']]
if len(mismatched):
    print('Columns with a registry dtype mismatch:')
    print(mismatched[['dtype', 'expected_data_type']])

# Map coverage by admin unit

In [ ]:
def _map_grid(n: int, ncols: int = 3, panel_size: float = 4.0):
    """Subplot grid capped at *ncols* columns, wrapping to more rows."""
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(panel_size * ncols, panel_size * nrows), squeeze=False
    )
    axes_flat = axes.flatten()
    for ax in axes_flat[n:]:
        ax.axis('off')
    return fig, axes_flat[:n]


# Columns worth checking first for a given entity_type, regardless of
# how they happen to rank by coverage variation -- these are the
# attributes most analyses actually depend on, so they should never get
# crowded out by a column that merely happens to vary more.
PRIORITY_COLUMNS_BY_ENTITY_TYPE = {
    'parcel': [
        'value',
        'land_value',
        'improvement_value',
        'last_sale_price',
        'address',
        'city',
        'owner_name',
        'owner_address',
        'use_group',
        'use_subgroup',
        'use_group_code',
    ],
}

if len(per_admin) > 1:
    boundary = op.get_admin(admin_ids, save_level, geom=True)

    # Columns that are 0% or 100% populated in every admin unit carry no
    # spatial signal -- a map of them is uniformly blank or uniformly full.
    # Focus on columns with genuine partial coverage instead.
    means = coverage.mean()
    partial_columns = means[(means > 0) & (means < 1)].index

    # Among those, surface the canonical output columns -- the registry's
    # named attributes, not id/source-tracking/evidence columns -- ranked
    # by how much their coverage *varies* across counties (the same spread
    # the violin plot above shows per column). A column whose coverage is
    # uniformly low everywhere is not informative on a map even though its
    # mean is low; a column whose coverage swings from one county to the
    # next is the one worth looking at.
    registry_names = set(get_attributes(entity_type).index)
    canonical_columns = [
        c for c in partial_columns if resolve_attribute_name(c) in registry_names
    ]
    variation = coverage[canonical_columns].std().sort_values(ascending=False)

    # Priority columns (that qualify above) always make the cut, in the
    # given order; the variation ranking only fills in the remaining
    # slots, and only pads the total past 6 if fewer than 6 priority
    # columns qualified.
    priority = [
        c
        for c in PRIORITY_COLUMNS_BY_ENTITY_TYPE.get(entity_type, [])
        if c in variation.index
    ]
    remaining = [c for c in variation.index if c not in priority]
    focus_columns = (priority + remaining)[: max(6, len(priority))]

    if len(focus_columns):
        fig, axes_flat = _map_grid(len(focus_columns))
        for ax, column in zip(axes_flat, focus_columns, strict=True):
            # A county missing from `coverage` never produced output for
            # this recipe at all -- distinct from a county that did, where
            # this column is uniformly null/placeholder, a real 0%. The
            # join leaves the former NaN; `missing_kwds` renders it as an
            # explicit hatch instead of a false 0%.
            merged = boundary.join((coverage[column] * 100).rename('value'))
            merged.plot(
                column='value',
                ax=ax,
                vmin=0,
                vmax=100,
                cmap=get_diverging_colormap('ember_ocean'),
                legend=True,
                legend_kwds={'label': '% meaningfully populated', 'shrink': 0.6},
                missing_kwds={
                    'color': DATA_AVAILABILITY_MISSING_COLOR,
                    'hatch': '///',
                    'edgecolor': 'white',
                },
            )
            boundary.boundary.plot(ax=ax, color='black', linewidth=0.2)
            ax.set_title(column)
            ax.axis('off')

        # geopandas doesn't add a legend entry for missing_kwds on a
        # continuous (colorbar) plot, so label the hatch once for the
        # whole figure rather than repeating a swatch on every subplot.
        no_data_patch = mpatches.Patch(
            facecolor=DATA_AVAILABILITY_MISSING_COLOR,
            hatch='///',
            edgecolor='white',
            label='no recipe output for this admin unit',
        )
        fig.legend(handles=[no_data_patch], loc='lower center', frameon=False)
    else:
        print('No canonical columns with partial, varying coverage to map.')
else:
    print('Only one admin unit loaded; skipping per-admin-unit coverage map.')

# Categorical value distributions

In [ ]:
categoricals = summarize_categoricals(combined, nmax=args.nmax_categories)
for column, counts in categoricals.items():
    print(f'\n{column} (top {min(len(counts), 10)} of {len(counts)}):')
    print(counts.head(10))

## Most common category by admin unit

For each canonical categorical column, which value is most common in
each admin unit.

Only columns whose most-common value actually *differs* across admin
units are shown. A column where the same category dominates everywhere
carries no spatial signal.

In [ ]:
def _top_category(series: pd.Series):
    mask = series.notna() & ~series.isin([0, ''])
    return series[mask].value_counts().idxmax() if mask.any() else None


def _bucket_and_truncate(series: pd.Series, nmax: int = 8, maxlen: int = 28):
    """Keep the *nmax* most frequent values, bucket the rest as 'Other'.

    A rare dominant value from a single small admin unit would otherwise
    earn its own legend swatch, and a long concatenated value (e.g. a
    pipe-joined use_group_combined) would blow up the legend's width --
    both keep the legend from actually being readable. True gaps (NaN)
    are left as NaN, not bucketed into 'Other'.
    """
    top_values = series.value_counts().head(nmax).index
    bucketed = series.where(series.isna() | series.isin(top_values), 'Other')
    return bucketed.map(
        lambda v: v if pd.isna(v) or len(v) <= maxlen else v[: maxlen - 1] + '…'
    )


if len(per_admin) > 1:
    registry_names = set(get_attributes(entity_type).index)
    modes = pd.DataFrame(
        {
            column: {
                # A column absent from one county's schema (e.g. present
                # only after pd.concat unioned columns in `combined`)
                # simply has no dominant category there.
                admin_id: _top_category(gdf[column]) if column in gdf else None
                for admin_id, gdf in per_admin.items()
            }
            for column in categoricals
            if resolve_attribute_name(column) in registry_names
        }
    )
    n_unique = modes.nunique()
    diverse_columns = n_unique[n_unique > 1].sort_values(ascending=False).index[:6]

    if len(diverse_columns):
        # `_map_grid` and `boundary` were already defined by the coverage
        # map above. Each panel here carries its own legend below the
        # map (unlike the shared-colorbar coverage map), so it needs a
        # much taller panel and a wide row gap to keep from colliding
        # with the title of the row underneath.
        fig, axes_flat = _map_grid(len(diverse_columns), panel_size=6.5)
        fig.subplots_adjust(hspace=0.6)
        for ax, column in zip(axes_flat, diverse_columns, strict=True):
            merged = boundary.join(_bucket_and_truncate(modes[column]).rename('value'))
            merged.plot(
                column='value',
                ax=ax,
                categorical=True,
                cmap='tab20',
                legend=True,
                legend_kwds={
                    'loc': 'upper center',
                    'bbox_to_anchor': (0.5, -0.02),
                    'ncols': 1,
                    'fontsize': 'small',
                },
                missing_kwds={'color': '#e0e0e0', 'hatch': '///'},
            )
            boundary.boundary.plot(ax=ax, color='black', linewidth=0.2)
            ax.set_title(column)
            ax.axis('off')
    else:
        print('No canonical categorical columns vary by admin unit.')
else:
    print('Only one admin unit loaded; skipping most-frequent-category map.')

# Geometry checks

In [ ]:
for admin_id, gdf in per_admin.items():
    report = check_geometry(gdf)
    # Polygon + MultiPolygon mixed together is normal (a multi-part
    # parcel and a single-part one are both valid) -- only flag other
    # geometry types (LineString, GeometryCollection, ...).
    geom_types = {t for t in report['geom_type_counts'].index if isinstance(t, str)}
    unexpected_types = geom_types - {'Polygon', 'MultiPolygon'}
    flagged = (
        report['n_null']
        or report['n_empty']
        or report['n_invalid']
        or bool(unexpected_types)
    )
    issues = [
        f'{report[key]} {label}'
        for key, label in (
            ('n_null', 'null'),
            ('n_empty', 'empty'),
            ('n_invalid', 'invalid'),
        )
        if report[key]
    ]
    summary = f'{admin_id}: {report["n_total"]:,d} rows'
    if issues:
        summary += ', ' + ', '.join(issues)
    print(summary)
    if flagged:
        parts = []
        for geom_type, n in report['geom_type_counts'].items():
            label = 'missing' if pd.isna(geom_type) else geom_type
            tag = ' (unexpected)' if geom_type in unexpected_types else ''
            parts.append(f'{int(n):,d} {label}{tag}')
        print('    geometry types: ' + ', '.join(parts))

## Optional: compare across sources

Only runs when `--compare_recipe_ids` is passed.

For each comparison recipe, rows are matched to the primary recipe on
the same physical entity:

- A shared id-like column, when both sides have one
- Otherwise, a spatial join

Every shared numeric column is then plotted as a log-log scatter
against a `y=x` reference line.

This is a best-effort default. A genuinely apples-to-apples comparison
between two different entity types may need a manual join instead.

In [ ]:
if args.compare_recipe_ids:
    id_keys = ('parcel_id_local', 'parcel_id_admin2')
    for compare_id in args.compare_recipe_ids:
        compare_frames = [
            op.get_entities(compare_id, admin_id=admin_id, geom=True, missing='warn')
            for admin_id in admin_ids
        ]
        compare_frames = [f for f in compare_frames if f is not None and len(f)]
        if not compare_frames:
            print(f'{compare_id}: no data for {admin_ids}, skipping.')
            continue
        compare_gdf = pd.concat(compare_frames)

        id_candidates = [
            c for c in id_keys if c in combined.columns and c in compare_gdf.columns
        ]
        if id_candidates:
            # 1:1 comparison only -- neither key is guaranteed unique
            # (a raw many-to-many merge on a duplicated key can blow up
            # combinatorially), so keep one row per key on each side,
            # the same way link_by_id's 'attributes' mode does.
            key = id_candidates[0]
            left = combined.dropna(subset=[key]).drop_duplicates(key)
            right = compare_gdf.dropna(subset=[key]).drop_duplicates(key)
            joined = left.merge(right, on=key, suffixes=('_left', '_right'))
        else:
            joined = gpd.sjoin(
                combined,
                compare_gdf,
                how='inner',
                predicate='intersects',
                lsuffix='left',
                rsuffix='right',
            )

        shared_columns = sorted(
            {c[: -len('_left')] for c in joined.columns if c.endswith('_left')}
            & {c[: -len('_right')] for c in joined.columns if c.endswith('_right')}
        )
        print(
            f'{compare_id}: {len(joined):,d} matched rows, '
            f'shared columns: {shared_columns}'
        )

        for column in shared_columns:
            left_col, right_col = f'{column}_left', f'{column}_right'
            if not pd.api.types.is_numeric_dtype(joined[left_col]):
                continue
            pair = joined[[left_col, right_col]].dropna()
            if pair.empty:
                continue
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.scatter(pair[left_col], pair[right_col], s=5, alpha=0.3)
            lims = [pair.min().min(), pair.max().max()]
            ax.plot(lims, lims, color='red', linewidth=1)
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.set_xlabel(f'{args.recipe_id}: {column}')
            ax.set_ylabel(f'{compare_id}: {column}')
            ax.set_title(column)
else:
    print('No --compare_recipe_ids given; skipping cross-source comparison.')

# Sample record

In [ ]:
pd.options.display.max_rows = combined.shape[1] + 5
op.inspect_table(combined, n=5)

In [ ]:
# Visual spot-check of one random entity, if the recipe has geometry.
show_random_entity(args.recipe_id, admin_ids[0])